In [1]:
import pandas as pd
import os
import json
import config as cfg
import numpy as np
import math
import matplotlib.pyplot as plt


C:\Users\d.c.macrae\AppData\Local\Temp\2\ipykernel_7100\3068521150.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was too old on your system - pyarrow 10.0.1 is the current minimum supported version as of this release.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## Seeds

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os


# FILEPATH: /appdata/RTDicom/Projectline_HNC_modelling/Users/Daniel MacRae/DL_NTCP_Multitox/inspect_data_splits.ipynb
csv_path = "//zkh/appdata/RTDicom/Projectline_HNC_modelling/Users/Daniel MacRae/NTCP_Multitox"

df_seeds = pd.read_csv(os.path.join(csv_path, "auc_results_04_08.csv"), index_col=0)
df_seeds['xero_test_auc']

name_train = "_train_auc"
name_test = "_test_auc"
toxicities = ["xero", "taste", "sticky", "dysphagia", "aspiration"]
bins = np.arange(0.5, 1, 0.005)

for tox in toxicities:
    mean_train = df_seeds[tox + name_train].mean()
    mean_test = df_seeds[tox + name_test].mean()

    plt.hist(df_seeds[tox + name_train], alpha=0.7, label=f'Train (mean = {mean_train:.3f})', bins=bins)
    plt.hist(df_seeds[tox + name_test], alpha=0.7, label=f' Test (mean = {mean_test:.3f})', bins=bins)

    plt.xlabel('AUC values')
    plt.ylabel('Frequency')
    plt.title(tox)
    plt.legend()
    plt.show()


C:\Users\MacRaeDC\AppData\Local\Temp\ipykernel_6088\2441675026.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Plot

In [3]:
csv_path = "datasets/MT_dataset/stratified_sampling_test_542.csv"
#csv_path = "//zkh/appdata/RTDicom/Projectline_HNC_modelling/Users/Daniel MacRae/NTCP_Multitox/stratified_sampling_test_02_09.csv"
df = pd.read_csv(csv_path, delimiter=";")
print(len(df))
df['PatientID'] = [str(item).zfill(cfg.patient_id_length) for item in df["PatientID"]]

train_val_df = df[df['Split']!="test"].copy()
test_df = df[df['Split']=="test"].copy()

toxicity_endpoint_features = cfg.toxicity_endpoint_features
strata_groups =  ['CT+C_available', 'CT_Artefact', 'Photons', 'Loctum2'] #+ toxicity_endpoint_features  # NEW MULTITOX: # (Stratified Sampling). Note: order does not matter.


plotting_columns = ['CT+C_available', 'CT_Artefact', 'Photons', 'Loctum2']  + cfg.toxicity_endpoint_features
strat_groups_title =  ['CT+C_available', 'CT_Artefact', 'Photons', 'Loctum2']  #  # #  # plotting_columns # 
#df_summary_statistics = pd.DataFrame(columns=["feature", "total_dataset", "training_datset", "testing_dataset"])

cols = 3
rows = math.ceil(len(plotting_columns) / cols)
bar_width = 0.2
normalise = True
colours = ['r', 'g', 'b', 'y', 'c']
names = ["Test", "Train_Val", "Train"]

fig,ax = plt.subplots(rows,cols,figsize=[4*cols,4*rows])
for i in range(len(plotting_columns)):
    col_name = plotting_columns[i]
    
    [row_idx, col_idx] = [int(i/cols),int(i % cols)]

    ax[row_idx, col_idx].set_title(col_name)

    max_keys = 0
    #indexer = train_val_df[col_name].sort_values().value_counts(normalize=normalise, dropna=False, sort=True).keys()

    for j, df_temp in enumerate([test_df, train_val_df]):
        value_counts = df_temp[col_name].value_counts(normalize=normalise, dropna=False)#.reindex(indexer)

        if len(value_counts.keys()) > max_keys:
            max_keys = len(value_counts.keys())

        value_counts.plot(kind='bar', ax=ax[row_idx, col_idx], color=colours[j], align='edge', position=j, width=bar_width, label=names[j])
       
    ax[row_idx, col_idx].set_xlabel(None)
    ax[row_idx, col_idx].axis(xmin = 0-2*bar_width, xmax=max_keys-1 + 2*bar_width)
    if normalise:
        ax[row_idx, col_idx].axis(ymax=1)

fig.suptitle(strat_groups_title, fontsize=15)
fig.tight_layout()
#fig.suptitle(title, fontsize = 13)
#plt.tight_layout()
#plt.savefig(label_images_save_dir + "/class_value_counts.png")
handles, labels = ax[0,0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right')

plt.tight_layout()
plt.legend()
plt.show()

1090


C:\Users\MacRaeDC\AppData\Local\Temp\ipykernel_6088\3303917831.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# See the train-val / test split demographics

In [2]:
import pandas as pd
import config as cfg

csv_path = "datasets/MT_dataset/stratified_sampling_test_542.csv"
#csv_path = "//zkh/appdata/RTDicom/Projectline_HNC_modelling/Users/Daniel MacRae/NTCP_Multitox/stratified_sampling_test_02_09.csv"
df = pd.read_csv(csv_path, delimiter=";")
print(len(df))
df['PatientID'] = [str(item).zfill(cfg.patient_id_length) for item in df["PatientID"]]

train_val_df = df[df['Split']!="test"].copy()
test_df = df[df['Split']=="test"].copy()

# Assuming df_train_val and df_test are already defined

# Create a DataFrame to store the value counts
value_counts_df = pd.DataFrame(columns=["Column", "Train_Val_Counts", "Test_Counts"])
columns_to_compare = ["Sex", "Age", "Loctum2", "Photons", "CT+C_available", "CT_Artefact"]  + cfg.toxicity_endpoint_features
columns_to_compare = cfg.toxicity_endpoint_features
# Iterate over the columns_to_compare list
for column in columns_to_compare:
    # Get the value counts for df_train_val and df_test
    #train_val_counts = df_train_val[column].value_counts()
    #test_counts = df_test[column].value_counts()
    
    print(column, "\nTrain\n", train_val_df[column].value_counts(), train_val_df[column].value_counts(normalize=True), 
          "\nTest\n", test_df[column].value_counts(), test_df[column].value_counts(normalize=True), "\n\n")


1090
Aspiration_M06 
Train
 Aspiration_M06
 0.0    721
-1.0     92
 1.0     59
Name: count, dtype: int64 Aspiration_M06
 0.0    0.826835
-1.0    0.105505
 1.0    0.067661
Name: proportion, dtype: float64 
Test
 Aspiration_M06
 0.0    175
-1.0     24
 1.0     19
Name: count, dtype: int64 Aspiration_M06
 0.0    0.802752
-1.0    0.110092
 1.0    0.087156
Name: proportion, dtype: float64 


Dysphagia_M06 
Train
 Dysphagia_M06
 0.0    634
 1.0    210
-1.0     28
Name: count, dtype: int64 Dysphagia_M06
 0.0    0.727064
 1.0    0.240826
-1.0    0.032110
Name: proportion, dtype: float64 
Test
 Dysphagia_M06
 0.0    160
 1.0     48
-1.0     10
Name: count, dtype: int64 Dysphagia_M06
 0.0    0.733945
 1.0    0.220183
-1.0    0.045872
Name: proportion, dtype: float64 


Sticky_M06 
Train
 Sticky_M06
 0.0    523
 1.0    260
-1.0     89
Name: count, dtype: int64 Sticky_M06
 0.0    0.599771
 1.0    0.298165
-1.0    0.102064
Name: proportion, dtype: float64 
Test
 Sticky_M06
 0.0    136
 1.0     61
-

In [3]:
UMCG = False


columns_to_compare = ["Sex", "Age", "Loctum2", 'T_stage', 'N_stage', 'P16', 'WHO', "Photons", "CT+C", "CT_Artefact"] # + cfg.toxicity_endpoint_features


if UMCG:
    csv_path = "datasets/MT_dataset/stratified_sampling_test_542.csv"
    columns_to_compare = cfg.toxicity_endpoint_features
else:
    csv_path = "datasets/MDACC_dataset/MDACC_features_present.csv"
    columns_to_compare = ['Xerostomia_BSL','Aspiration_BSL', 'Sticky_BSL', 'Taste_BSL', 'Dysphagia_BSL']

#csv_path = "//zkh/appdata/RTDicom/Projectline_HNC_modelling/Users/Daniel MacRae/NTCP_Multitox/stratified_sampling_test_02_09.csv"
df = pd.read_csv(csv_path, delimiter=";")
print(len(df))
df['PatientID'] = [str(item).zfill(cfg.patient_id_length) for item in df["PatientID"]]



# Iterate over the columns_to_compare list
train_val_df = df[df['Split']!="test"].copy()
test_df = df[df['Split']=="test"].copy()

mean_SD_columns = ["Age"]

for column in columns_to_compare:
    if column not in mean_SD_columns:
        
        value_counts = test_df[column].value_counts(normalize=False)
        
        value_counts_norm = test_df[column].value_counts(normalize=True)
    else:
        value_counts = test_df[column].mean()
        value_counts_norm = test_df[column].std()
        print(column)

    print(value_counts)
    print(value_counts_norm)
    print('\n\n')
    # Get the value counts for df_train_val and df_test
    #train_val_counts = df_train_val[column].value_counts()
    #test_counts = df_test[column].value_counts()
    
    # print(column, "\nTrain\n", train_val_df[column].value_counts(), train_val_df[column].value_counts(normalize=True), 
    #       "\nTest\n", test_df[column].value_counts(), test_df[column].value_counts(normalize=True), "\n\n")

359
Xerostomia_BSL
0.0    314
1.0     45
Name: count, dtype: int64
Xerostomia_BSL
0.0    0.874652
1.0    0.125348
Name: proportion, dtype: float64



Aspiration_BSL
0.0    346
1.0     13
Name: count, dtype: int64
Aspiration_BSL
0.0    0.963788
1.0    0.036212
Name: proportion, dtype: float64



Sticky_BSL
0.0    324
1.0     35
Name: count, dtype: int64
Sticky_BSL
0.0    0.902507
1.0    0.097493
Name: proportion, dtype: float64



Taste_BSL
0.0    336
1.0     23
Name: count, dtype: int64
Taste_BSL
0.0    0.935933
1.0    0.064067
Name: proportion, dtype: float64



Dysphagia_BSL
0.0    322
1.0     37
Name: count, dtype: int64
Dysphagia_BSL
0.0    0.896936
1.0    0.103064
Name: proportion, dtype: float64





# get CT_Artefact original ranking

In [4]:
train_val_df

if UMCG:
    df_CT_artefact = pd.read_csv("//zkh/appdata/RTDicom/Projectline_HNC_modelling/Users/HungChu/Figures/PRI2MA/metal_artifacts.csv", delimiter=";")
else:
     df_CT_artefact = pd.read_csv("//zkh/appdata/RTDicom/Projectline_HNC_modelling/Users/HungChu/Figures/MDACC/metal_artifacts.csv", delimiter=";")
df_CT_artefact['PatientID'] = [str(item).zfill(cfg.patient_id_length) for item in df_CT_artefact["PatientID"]]

FileNotFoundError: [Errno 2] No such file or directory: '//zkh/appdata/RTDicom/Projectline_HNC_modelling/Users/HungChu/Figures/MDACC/metal_artifacts.csv'

In [46]:
df_CT_artefact

,PatientID,CT_Artefact,Wie?,CT_Artefact Suzanne 1st round (only 0 or 1),Opmerkingen,Unnamed: 5,Unnamed: 6,Unnamed: 7,Summary,0,1,2,Total
0,1014175846,0,HN,0,NaN,NaN,NaN,NaN,HN,32,66,69,167.0
1,1019236881,0,HN,0,NaN,NaN,NaN,NaN,SdV,24,77,67,168.0
2,1022096596,1,HN,1,NaN,NaN,NaN,NaN,HC,29,67,74,170.0
3,1027008628,0,HN,0,NaN,NaN,NaN,NaN,Total,85,210,210,505.0
4,1029341402,0,HN,0,NaN,NaN,NaN,NaN,%,"16,83168317","41,58415842","41,58415842",100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
500,9626079921,2,HC,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
501,9643148771,2,HC,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
502,9681706785,0,HC,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
503,9878102359,1,HC,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
CT_Art_merged_df = test_df.merge(df_CT_artefact, on='PatientID', how='left')


In [48]:
print(CT_Art_merged_df.CT_Artefact_y.value_counts())

print(CT_Art_merged_df.CT_Artefact_y.value_counts(normalize=True))

CT_Artefact_y
1    153
2    149
0     57
Name: count, dtype: int64
CT_Artefact_y
1    0.426184
2    0.415042
0    0.158774
Name: proportion, dtype: float64


In [49]:
print(CT_Art_merged_df.CT_Artefact_x.value_counts())

print(CT_Art_merged_df.CT_Artefact_x.value_counts(normalize=True))

CT_Artefact_x
1    153
2    149
0     57
Name: count, dtype: int64
CT_Artefact_x
1    0.426184
2    0.415042
0    0.158774
Name: proportion, dtype: float64


# Try to find treatment modes

In [7]:
train_val_df = df[df['Split']!="test"].copy()
test_df = df[df['Split']=="test"].copy()

#df_CITOR_REDCAP = pd.read_excel("//zkh/appdata/RTDicom/Projectline_HNC_modelling/Users/Daniel MacRae/CITOR_REDCAP_clinical_data_important_variables_combined.xlsx")
# FOR DRE:

if UMCG:
    df_CITOR_REDCAP = pd.read_excel("c:/Users/d.c.macrae/Documents/CITOR_REDCAP_clinical_data_important_variables_combined.xlsx")
    df_CITOR_REDCAP['PatientID'] = [str(item).zfill(cfg.patient_id_length) for item in df_CITOR_REDCAP["UMCG"]]
else:
    df_CITOR_REDCAP = pd.read_excel("c:/Users/d.c.macrae/Documents/MDACC_database-clean_Daniel.xlsx")
    df_CITOR_REDCAP['PatientID'] = [str(item).zfill(cfg.patient_id_length) for item in df_CITOR_REDCAP["newID"]]

#df_CITOR_REDCAP['PatientID'] = [str(item).zfill(cfg.patient_id_length) for item in df_CITOR_REDCAP["UMCG"]]

In [14]:
df_CITOR_REDCAP["Treatment technique"]

0      IMRT
1      VMAT
2      VMAT
3      VMAT
4      IMRT
       ... 
402    VMAT
403    VMAT
404    VMAT
405    VMAT
406    IMPT
Name: Treatment technique, Length: 407, dtype: object

In [8]:
df_CITOR_REDCAP.columns

Index(['ID', 'newID', 'AGE1', 'CT_Artefact', 'Year', 'BaselineWeight',
       'x3_6MonthsWeight', 'x12MonthsWeight', 'Sex',
       'ConcurrentTherapyIncludingChemoOrTargetTherapy',
       ...
       'Submandibular_RDmean_DLC', 'SupraglotticDmean_DLC', 'BrainDmean_DLC',
       'MandibleDmean_DLC', 'BrainstemDmean_DLC', 'Submandibular_Dmean_DLC',
       'Parotid_Dmean', 'CT_Artefact.1', 'P16', 'PatientID'],
      dtype='object', length=600)

In [16]:
mdacc_merged = df.merge(df_CITOR_REDCAP, on='PatientID', how='left')
mdacc_merged["Treatment technique"].value_counts()

Treatment technique
VMAT         205
IMRT          66
IMPT          62
IMRT/VMAT     22
3-D plan       4
Name: count, dtype: int64

In [17]:
mdacc_merged["Treatment technique"].value_counts(normalize=True)

Treatment technique
VMAT         0.571031
IMRT         0.183844
IMPT         0.172702
IMRT/VMAT    0.061281
3-D plan     0.011142
Name: proportion, dtype: float64

In [9]:
train_merged = train_val_df.merge(df_CITOR_REDCAP, on='PatientID', how='left')

test_merged = test_df.merge(df_CITOR_REDCAP, on='PatientID', how='left')

In [13]:
df_CITOR_REDCAP.columns

Index(['UMCG', 'DB', 'RTSTART', 'GESLACHT', 'LEEFTIJD', 'Technique',
       'Technique_REDCAP', 'Technique_check', 'Check2', 'Inplancode', 'LOCTUM',
       'Loctum2', 'ROKEN', 'TSTAD_DEF', 'Histology', 'N_stage', 'M_stage',
       'P16', 'Doelgebied', 'Seq_Rad', 'Modality', 'Modality_adjusted', 'WHO',
       'Alcohol', 'HN35_Xerostomia_BSL', 'HN35_Xerostomia_W01',
       'HN35_Xerostomia_M06', 'HN35_Xerostomia_M12', 'HN35_Xerostomia_M18',
       'HN35_Xerostomia_M24', 'HN35_Aspiration_BSL', 'HN35_Aspiration_W01',
       'HN35_Aspiration_M06', 'HN35_Aspiration_M12', 'HN35_Aspiration_M18',
       'HN35_Aspiration_M24', 'HN35_Sticky_BSL', 'HN35_Sticky_W01',
       'HN35_Sticky_M06', 'HN35_Sticky_M12', 'HN35_Sticky_M18',
       'HN35_Sticky_M24', 'HN35_Taste_BSL', 'HN35_Taste_W01', 'HN35_Taste_W02',
       'HN35_Taste_W03', 'HN35_Taste_M06', 'HN35_Taste_M12', 'HN35_Taste_M18',
       'HN35_Taste_M24', 'DYSFAGIE_UMCG_BSL', 'DYSFAGIE_UMCG_W01',
       'DYSFAGIE_UMCG_m06', 'DYSFAGIE_UMCG_m12'

In [19]:
train_merged.Technique.value_counts()
train_merged['Technique_Dan'] = train_merged['Technique']
train_merged['Technique_Dan'][train_merged['Technique_Dan'] == 0] = train_merged['Check2']
train_merged['Technique_Dan'].value_counts()

C:\Users\d.c.macrae\AppData\Local\Temp\2\ipykernel_10280\576164381.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  train_merged['Technique_Dan'][train_merged['Technique_Dan'] == 0] = train_merged['Check2']
C:\Users\d.c.macrae\AppData\Loc

Technique_Dan
IMRT            387
VMAT            294
Protons         120
3D-CRT           56
IMRT + VMAT       6
Stereotactic      2
Name: count, dtype: int64

In [24]:
train_merged['Technique_Dan'].value_counts(normalize=True)

Technique_Dan
IMRT            0.447399
VMAT            0.339884
Protons         0.138728
3D-CRT          0.064740
IMRT + VMAT     0.006936
Stereotactic    0.002312
Name: proportion, dtype: float64

In [21]:
test_merged.Technique.value_counts()
test_merged['Technique_Dan'] = test_merged['Technique']
test_merged['Technique_Dan'][test_merged['Technique_Dan'] == 0] = test_merged['Check2']
test_merged['Technique_Dan'].value_counts()

C:\Users\d.c.macrae\AppData\Local\Temp\2\ipykernel_10280\3220224775.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  test_merged['Technique_Dan'][test_merged['Technique_Dan'] == 0] = test_merged['Check2']
C:\Users\d.c.macrae\AppData\Local

Technique_Dan
IMRT           90
VMAT           80
Protons        30
3D-CRT         13
IMRT + VMAT     5
Name: count, dtype: int64

In [23]:
test_merged['Technique_Dan'].value_counts(normalize=True)

Technique_Dan
IMRT           0.412844
VMAT           0.366972
Protons        0.137615
3D-CRT         0.059633
IMRT + VMAT    0.022936
Name: proportion, dtype: float64

In [13]:
test_merged.Technique.value_counts()

Technique
IMRT           90
VMAT           80
Protons        23
3D-CRT         13
0               7
IMRT + VMAT     5
Name: count, dtype: int64

In [33]:
train_merged.ROKEN.value_counts(normalize=False)

ROKEN
Ja, nog steeds     414
Ja, in verleden    348
Nee                110
Name: count, dtype: int64

In [34]:
test_merged.ROKEN.value_counts(normalize=False)

ROKEN
Ja, nog steeds     101
Ja, in verleden     92
Nee                 25
Name: count, dtype: int64

In [ ]:
df.columns

Index(['PatientID', 'Sex', 'Age', 'CT+C_available', 'CT_Artefact', 'Loctum2',
       'Photons', 'Xerostomia_BSL', 'Xerostomia_W01', 'Aspiration_BSL',
       'Aspiration_W01', 'Sticky_BSL', 'Sticky_W01', 'Taste_BSL', 'Taste_W01',
       'Dysphagia_BSL', 'Dysphagia_W01', 'Xerostomia_M06', 'Xerostomia_M12',
       'Xerostomia_M18', 'Aspiration_M06', 'Aspiration_M12', 'Aspiration_M18',
       'Sticky_M06', 'Sticky_M12', 'Sticky_M18', 'Taste_M06', 'Taste_M12',
       'Taste_M18', 'Dysphagia_M06', 'Dysphagia_M12', 'Dysphagia_M18',
       'Parotid_L_meandose', 'Parotid_R_meandose', 'Parotid_meandose',
       'Submandibular_L_meandose', 'Submandibular_R_meandose',
       'Submandibular_meandose', 'PCM_Sup_meandose', 'PCM_Med_meandose',
       'PCM_Inf_meandose', 'Crico_meandose', 'Supraglottic_meandose',
       'OralCavity_Ext_meandose', 'BuccalMucosa_L_meandose',
       'BuccalMucosa_R_meandose', 'BuccalMucosa_meandose',
       'TongueTop_meandose', 'External_meandose', 'PCM_meandose',
      

In [ ]:
import pandas as pd
# Create a copy of the original dataframe
df = pd.read_csv(csv_path, delimiter=";")
df['PatientID'] = [str(item).zfill(cfg.patient_id_length) for item in df["PatientID"]]
df_copy = df.copy()


per_tox_subfolder = os.path.join(cfg.data_dir, "per_toxicity_csvs")
os.makedirs(per_tox_subfolder, exist_ok=True)
# Iterate over each column name in the list
for column_name in cfg.toxicity_endpoint_features:
    # Drop rows with -1 in the current column
    df_copy = df[df[column_name] != -1].copy()

    # Save the resulting dataframe
    df_copy.to_csv(os.path.join(per_tox_subfolder, f'{column_name}_02_21.csv'), index=False, sep=";")

    

    #print(df_copy[column_name].value_counts())
    #print(df[column_name].value_counts())
